# Campus Control System — Assignment 1
Introduction to Data Science

This notebook demonstrates the four required components: the OOP domain model, the timetable conflict engine, the polymorphic space survival game (including a genuinely interactive turn loop), and the modular MissionControl coordinator with save/resume.

In [ ]:
import sys, io, builtins
sys.path.insert(0, '.')
from models import (Person, Student, Instructor, TimeSlot, CourseSection,
                     Planet, Mission, EventRecord, build_sections_from_records)
from exceptions import InvalidTimeError, DuplicateIdentifierError, InvalidRecordError, InvalidActionError
from timetable import ConflictEngine, print_timetable_grid, print_conflict_stats
from game import Game, ExplorerCraft, DefenderCraft, EVENTS
from main import MissionControl, SAMPLE_RECORDS, INVALID_RECORD

def run_with_input(func, scripted_text):
    
    lines = iter(scripted_text.splitlines())
    real_input = builtins.input

    def fake_input(prompt=''):
        print(prompt, end='')
        try:
            line = next(lines)
        except StopIteration:
            raise EOFError('scripted input exhausted')
        print(line)
        return line

    builtins.input = fake_input
    try:
        return func()
    finally:
        builtins.input = real_input

## Question 1 — OOP Domain Model and Data Import

`Person` is the base class, with `Student` and `Instructor` overriding `role_summary()` to show different output. `TimeSlot` validates that the end time is after the start and stores minute intervals. `CourseSection` stores enrolled students in a set. The loader builds `CourseSection` objects from a list of ten-plus record dictionaries and rejects invalid ones using `InvalidTimeError` and `DuplicateIdentifierError`.

In [2]:
alice = Student('Alice Khan', '24I-101', major='Data Science')
bilal = Instructor('Bilal Ahmed', 'I-07', department='Computer Science')
print(alice.role_summary())
print(bilal.role_summary())
print(alice.summary())
print(bilal.summary())

slot = TimeSlot('Monday', '09:00', '10:30')
print(slot, '-> interval in minutes:', slot.interval())

try:
    TimeSlot('Monday', '10:00', '09:00')
except InvalidTimeError as exc:
    print('Caught expected error:', exc)

Student Alice Khan (24I-101) majoring in Data Science, enrolled in 0 section(s).
Instructor Bilal Ahmed (I-07) of Computer Science, teaching 0 section(s).
Student Alice Khan (24I-101) majoring in Data Science, enrolled in 0 section(s).
Instructor Bilal Ahmed (I-07) of Computer Science, teaching 0 section(s).
Monday 09:00-10:30 -> interval in minutes: (540, 630)
Caught expected error: End time must be later than start time for Monday 10:00-09:00


In [3]:
records = SAMPLE_RECORDS + [INVALID_RECORD]
sections, rejected = build_sections_from_records(records)
print(f'Loaded {len(sections)} valid sections out of {len(records)} records')
print(f'Rejected {len(rejected)} record(s):')
for record, reason in rejected:
    print(' -', record.get('course_code', '?'), '->', reason)
print()
for section in sections[:5]:
    print(section, '|', section.summary())

Loaded 11 valid sections out of 12 records
Rejected 1 record(s):
 - BAD -> Unknown day: Someday

IDS-A Intro to Data Science @ R-201 [Monday 09:00-10:30] (2/2) | Section IDS-A: 2/2 enrolled at R-201
AI-A Artificial Intelligence @ R-305 [Monday 09:00-10:00] (1/30) | Section AI-A: 1/30 enrolled at R-305
DB-B Databases @ R-201 [Tuesday 10:00-11:00] (2/25) | Section DB-B: 2/25 enrolled at R-201
DB-A Databases @ R-102 [Tuesday 10:00-11:30] (1/25) | Section DB-A: 1/25 enrolled at R-102
AI-B Artificial Intelligence @ R-306 [Wednesday 09:00-10:30] (1/30) | Section AI-B: 1/30 enrolled at R-306


## Question 2 — Timetable Conflict Engine

`ConflictEngine` compares every pair of sections and detects student, instructor, room, capacity, and same-course section conflicts. A lambda sorts sections by day and start time, a nested helper function formats report lines, and two different report functions (`summary_report`, `detailed_report`) are produced by passing a formatter function into `build_report`, which also takes default and keyword arguments.

**Time complexity:** pairwise conflict detection compares every pair of sections, which is O(n^2) for n sections. This can be improved by first grouping sections by day and by room/instructor/student key, so only sections that already share a day and a resource are compared, cutting the number of comparisons dramatically for large timetables.

In [4]:
engine = ConflictEngine(sections)
conflicts = engine.scan_timetable()
print(engine.summary_report(conflicts))
print()
print(engine.detailed_report(conflicts))
print()
print_conflict_stats(conflicts, engine)

=== Summary ===
student: IDS-A vs AI-A
section: DB-B vs DB-A
Total: 2

=== Detailed ===
!STUDENT IDS-A <-> AI-A on Monday (540, 600)
!SECTION DB-B <-> DB-A on Tuesday (600, 660)

CONFLICT SUMMARY
Student: 1 Section: 1


In [5]:
print('Timetable grid for student 24I-101:')
print_timetable_grid(engine.sorted_sections(), conflicts, focus_id='24I-101', focus_type='student')
print()
print('Schedule lookup for an unknown student id (must not crash):')
print_timetable_grid(engine.sorted_sections(), conflicts, focus_id='99I-999', focus_type='student')

Timetable grid for student 24I-101:
No sections found for student '24I-101'. Check the id and try again.

Schedule lookup for an unknown student id (must not crash):
No sections found for student '99I-999'. Check the id and try again.


## Question 3 — Polymorphic Space Survival Game

`ExplorerCraft` and `DefenderCraft` both inherit from `Spacecraft` and the `Auditable` mixin (multiple inheritance), and each overrides `travel()` differently. `Game.play()` is the real, interactive turn loop: it prints the HUD, shows a numbered action menu, reads the player's choice with `input()`, validates it, applies the effect, fires a random event through the `EVENTS` dictionary, and checks win/loss — exactly like the terminal sample in the assignment brief. Below we drive that real `play()` loop with a scripted sequence of keystrokes (`run_with_input`) so the notebook shows genuine gameplay output rather than a test harness.

In [6]:
print('Registered events:', list(EVENTS.keys()))

for craft_cls in (ExplorerCraft, DefenderCraft):
    craft = craft_cls('Test-' + craft_cls.__name__, 'Earth')
    craft.travel('Kepler-1', 2)
    print(craft_cls.__name__, '-> fuel after travel:', craft.fuel, '| max_health:', craft.max_health)

Registered events: ['resource_gain', 'fuel_loss', 'oxygen_loss', 'hull_damage', 'system_malfunction', 'beneficial_repair', 'energy_surge', 'solar_storm']
ExplorerCraft -> fuel after travel: 78 | max_health: 100
DefenderCraft -> fuel after travel: 74 | max_health: 130


In [ ]:

keystrokes = "\n".join([
    '1', 'Kepler-1',     
    '2',                   
    '3', 'metal',         
    '1', 'Kepler-2',       
    '5', 'M1',            
    '4', '10',           
    '1', 'Nexus',        
    '5', 'M2',            
    '1', 'Finale',        
    '5', 'M3',           
    '11',                  
]) + "\n"

craft = ExplorerCraft('Odyssey', 'Earth')
game = Game(craft, seed=7)
won, lost = run_with_input(game.play, keystrokes)
print()
print('Result:', 'WON' if won else ('LOST' if lost else 'IN PROGRESS'))

Welcome aboard! Enter the number for an action each turn.
================ SPACE SURVIVAL ================
Turn: 0/20 Craft: Odyssey
Odyssey @ Earth | Hull 100/100 Fuel 80/80 Oxygen 100/100 Energy 60/60 Cargo 0/10
Missions: 0/3
[Earth*] -- [???] -- [???] -- [???] -- [???]
1 Travel      2 Scan          3 Collect      4 Repair
5 Accept Msn  6 Abandon Msn   7 Upgrade      8 View Missions
9 Inventory   10 Event Log    11 Quit
Choose action (1-11): 1
Reachable from Earth : Kepler-1
Destination planet: Kepler-1
EVENT: Friendly drone patched the hull -> hull +7
================ SPACE SURVIVAL ================
Turn: 1/20 Craft: Odyssey
Odyssey @ Kepler-1 | Hull 100/100 Fuel 78/80 Oxygen 99/100 Energy 60/60 Cargo 0/10
Missions: 0/3
[Earth] -- [Kepler-1*] -- [???] -- [???] -- [???]
1 Travel      2 Scan          3 Collect      4 Repair
5 Accept Msn  6 Abandon Msn   7 Upgrade      8 View Missions
9 Inventory   10 Event Log    11 Quit
Choose action (1-11): 2
Scan results: {'metal': 4, 'water': 2}
E

In [8]:
print('Audit trail (mission log of events):')
for line in craft.show_audit_trail():
    print(line)

try:
    game.action_repair(-5)
except InvalidActionError as exc:
    print('\nCaught expected error on invalid repair:', exc)

try:
    game.action_travel('Kepler-1')
except InvalidActionError as exc:
    print('Caught expected error on impossible travel from Finale:', exc)

print()
print('The play() loop never crashes on bad input either -- invalid menu choices,')
print('non-numeric repair amounts, and unknown mission ids are all caught and reported.')
bad_keystrokes = "\n".join(['99', 'abc', '4', 'notanumber', '5', 'BOGUS', '11']) + "\n"
safety_craft = ExplorerCraft('Safety-Check', 'Earth')
safety_game = Game(safety_craft, seed=1)
run_with_input(safety_game.play, bad_keystrokes)
print('No exceptions raised -- the game handled every bad input gracefully.')

Audit trail (mission log of events):
Turn 1: [beneficial_repair] Friendly drone patched the hull -> hull +7
Turn 2: [energy_surge] Solar flare recharged the batteries -> energy +15
Turn 3: [resource_gain] Found metal in deep space -> cargo +1 (metal)
Turn 4: [fuel_loss] Fuel leak detected -> fuel -10
Turn 5: [resource_gain] Found crystal in deep space -> cargo +1 (crystal)
Turn 6: [hull_damage] Micrometeorite impact -> hull -6
Turn 7: [fuel_loss] Fuel leak detected -> fuel -11

Caught expected error on invalid repair: Repair amount must be positive
Caught expected error on impossible travel from Finale: No direct route from Finale to Kepler-1

The play() loop never crashes on bad input either -- invalid menu choices,
non-numeric repair amounts, and unknown mission ids are all caught and reported.
Welcome aboard! Enter the number for an action each turn.
================ SPACE SURVIVAL ================
Turn: 0/20 Craft: Safety-Check
Safety-Check @ Earth | Hull 100/100 Fuel 80/80 Oxygen 

## Question 4 — Modular Integration and Command System

`MissionControl` owns the active timetable and game state and delegates to `ConflictEngine` and `Game`. Its `commands` dictionary maps command names (`load`, `schedules`, `conflicts`, `startgame`, `logs`, `save`, `resume`, `status`, `exit`) to bound methods, forming the menu shown by `run_menu()`. Choosing `startgame` from that menu prompts for a craft type and seed and then immediately calls `Game.play()`, so the game is genuinely playable from the command line, not just instantiated. `summary()`/`status()` are implemented across unrelated classes (`Person`, `CourseSection`, `Mission`, `Game`, `MissionControl`) to demonstrate a shared polymorphic interface. Game state is saved to and resumed from JSON — including the random-number generator's exact state — so a resumed game continues with the same sequence of events it would have had uninterrupted.

In [9]:
mc = MissionControl()
print('Available commands:', list(mc.commands.keys()))
mc.load_data()
print()
mc.run_conflict_analysis()
print()
# 'startgame' prompts for craft type + seed, then plays for real -- scripted here as a demo.
startgame_keystrokes = "\n".join([
    'explorer', '7',
    '1', 'Kepler-1', '2', '1', 'Kepler-2', '5', 'M1', '1', 'Nexus', '5', 'M2', '1', 'Finale', '5', 'M3',
    '11',
]) + "\n"
run_with_input(mc.start_and_play_interactive, startgame_keystrokes)

Available commands: ['load', 'schedules', 'conflicts', 'startgame', 'logs', 'save', 'resume', 'status', 'exit']
Loaded 11 section(s), rejected 1 record(s).
  rejected BAD: Unknown day: Someday

=== Summary ===
student: IDS-A vs AI-A
section: DB-B vs DB-A
Total: 2
CONFLICT SUMMARY
Student: 1 Section: 1

Craft type (explorer/defender) [explorer]: explorer
Random seed (whole number) [7]: 7
Game started with ExplorerCraft seed=7
Welcome aboard! Enter the number for an action each turn.
================ SPACE SURVIVAL ================
Turn: 0/20 Craft: Odyssey
Odyssey @ Earth | Hull 100/100 Fuel 80/80 Oxygen 100/100 Energy 60/60 Cargo 0/10
Missions: 0/3
[Earth*] -- [???] -- [???] -- [???] -- [???]
1 Travel      2 Scan          3 Collect      4 Repair
5 Accept Msn  6 Abandon Msn   7 Upgrade      8 View Missions
9 Inventory   10 Event Log    11 Quit
Choose action (1-11): 1
Reachable from Earth : Kepler-1
Destination planet: Kepler-1
EVENT: Friendly drone patched the hull -> hull +7
==========

In [ ]:
mc.view_logs()
print()
mc.save_game()
print()

run_with_input(mc.resume_game, 'n\n')
print()
print(mc.status())

Turn 1: [beneficial_repair] Friendly drone patched the hull -> hull +7
Turn 2: [energy_surge] Solar flare recharged the batteries -> energy +15
Turn 3: [resource_gain] Found metal in deep space -> cargo +1 (metal)
Turn 4: [fuel_loss] Fuel leak detected -> fuel -10
Turn 5: [resource_gain] Found crystal in deep space -> cargo +1 (crystal)

Saved to f:\Campus_Control_System_IDS\Campus_Control_System\save_state.json

Resumed game at turn 5, location Finale
Resume playing now? (y/n): n

MissionControl: 11 section(s) loaded, Game turn 5/20, craft at Finale, 3/3 missions done


In [11]:
print('Polymorphism through the shared summary() interface:')
for obj in [alice, bilal, sections[0], mc.game.missions[0], mc.game]:
    print(type(obj).__name__, '->', obj.summary())

Polymorphism through the shared summary() interface:
Student -> Student Alice Khan (24I-101) majoring in Data Science, enrolled in 0 section(s).
Instructor -> Instructor Bilal Ahmed (I-07) of Computer Science, teaching 0 section(s).
CourseSection -> Section IDS-A: 2/2 enrolled at R-201
Mission -> Mission M1 (done): Deliver crystal sample
Game -> Game turn 5/20, craft at Finale, 3/3 missions done
